

# Transfer Learning on E. coli to S. aureus

#### Mukundan Thanigaivelan, <thanigam@union.edu>

#### June 12, 2026

This notebook contains work on training Denoising Autoencoders on E. coli RNA-Seq data, and fine-tuning these models on S. aureus data to assess if transfer learning helps the model learn S. aureus pathways.

#### Outline

0. Before anything, we connect to the GitHub repository.
1. First we'll load classes from adage and modules for plotting, etc.
2. Next we'll take a look at the training data!
3. Train some models with hyperparameters from previous search.
4. Plot training loss over epochs.
5. Plot weight distributions of final models.

## 0. Connect to GitHub

In [1]:
!git clone https://github.com/Mukundan-T/seqADAGE.git

Cloning into 'seqADAGE'...
remote: Enumerating objects: 722, done.
remote: Counting objects: 100% (196/196), done.
remote: Compressing objects: 100% (156/156), done.
remote: Total 722 (delta 126), reused 50 (delta 40), pack-reused 526 (from 2)
Receiving objects: 100% (722/722), 56.73 MiB | 35.75 MiB/s, done.
Resolving deltas: 100% (376/376), done.


In [2]:
%cd seqADAGE/Py/muk_transfer_learning

/content/seqADAGE/Py/muk_transfer_learning


In [8]:
!pip install -qq tensorflow keras statsmodels seaborn keras_tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 1.9 MB/s eta 0:00:00


## 1. Loading classes & modules

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [20]:
# ADAGE
from adage import Adage as ad
from adage import SeqAdage

# Data Analysis
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Miscellaneous
import time
import json
import tensorflow as tf

In [10]:
# check CPU and GPU available in runtime
print("Num CPUs Available: ", len(tf.config.list_physical_devices('CPU')))
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.test.is_built_with_cuda())

Num CPUs Available:  1
Num GPUs Available:  1
True


## 2. E. coli models

The hyperparameter search landed on the following:

* Units: 40
* Dropout: 0.30000000000000004
* Activation 1: Sigmoid
* Activation 2: CeLU
* Shuffle: True
* Initialization: glorot_normal
* kl1: 0.11
* kl2: 0.30000000000000004
* lr: 0.091
* bs: 10
* mm: 0.8
* Tied: True

### 2a. Find and save optimal model hyperparameters

In [11]:
# Load filtered and re-normalized data
filename = '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/ecmg_lcn01_medmean_normal_muk.csv'

filtered_df = pd.read_csv(filename, index_col = 0)
filtered_df.shape

(3608, 10233)

In [12]:
# Find optimal hyperparameter values
model_trainer = SeqAdage.SeqAdage(filename)
best_hps, tuner = model_trainer.tune_model(seed = 42)

Trial 90 Complete [00h 00m 29s]
val_loss: 0.008467385545372963

Best val_loss So Far: 0.004861992318183184
Total elapsed time: 00h 22m 31s
Results summary
Results in /content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/ecmg_lcn01_medmean_normal_muk_42
Showing 10 best trials
Objective(name="val_loss", direction="min")

Trial 0050 summary
Hyperparameters:
units: 40
dropout: 0.30000000000000004
act1: sigmoid
act2: celu
shuffle: True
init: glorot_normal
kl1: 0.11
kl2: 0.30000000000000004
lr: 0.091
bs: 10
mm: 0.8
tied: 1
tuner/epochs: 50
tuner/initial_epoch: 17
tuner/bracket: 3
tuner/round: 3
tuner/trial_id: 0046
Score: 0.004861992318183184

Trial 0051 summary
Hyperparameters:
units: 10
dropout: 0.7000000000000001
act1: celu
act2: celu
shuffle: True
init: glorot_uniform
kl1: 0.1
kl2: 0.35000000000000003
lr: 0.041
bs: 5
mm: 0.8
tied: 1
tuner/epochs: 50
tuner/initial_epoch: 17
tuner/bracket: 3
tuner/round: 3
tuner/trial_id: 0047
Score: 0.005941049195826054

Trial 0046 summary
Hyperparam

In [14]:
# Optimal hyperparameters
best_hps[0].values

{'units': 40,
 'dropout': 0.30000000000000004,
 'act1': 'sigmoid',
 'act2': 'celu',
 'shuffle': True,
 'init': 'glorot_normal',
 'kl1': 0.11,
 'kl2': 0.30000000000000004,
 'lr': 0.091,
 'bs': 10,
 'mm': 0.8,
 'tied': 1,
 'tuner/epochs': 50,
 'tuner/initial_epoch': 17,
 'tuner/bracket': 3,
 'tuner/round': 3,
 'tuner/trial_id': '0046'}

In [23]:
# Save optimal model parameters
optimal_hps = {param: val for param, val in best_hps[0].values.items() if not param.startswith('tuner')}
with open('/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/ec_best_model_hps.json', "w") as f:
  json.dump(optimal_hps, f)

### 2b. Train model with optimal hyperparameters

In [26]:
# Load hyperparameters
with open('/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/ec_best_model_hps.json', "r") as f:
  optimal_hps = json.load(f)

# Train optimal model
sa = SeqAdage.SeqAdage(
  filename,
  enc_dim = optimal_hps['units'],
  kl1 = optimal_hps['kl1'],
  kl2 = optimal_hps['kl2'],
  act = optimal_hps['act1'],
  act2 = optimal_hps['act2'],
  tied = optimal_hps['tied'],
  epochs = 50,
  init = optimal_hps['init'],
  batch_size = optimal_hps['bs'],
  dropout = optimal_hps['dropout'],
  mm = optimal_hps['mm'],
  lr = optimal_hps['lr']
)
ec_model = sa.train_model()

Epoch 1/50
921/921 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.0266 - val_loss: 0.0201
Epoch 2/50
921/921 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0203 - val_loss: 0.0201
Epoch 3/50
921/921 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0202 - val_loss: 0.0198
Epoch 4/50
921/921 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0200 - val_loss: 0.0194
Epoch 5/50
921/921 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0193 - val_loss: 0.0189
Epoch 6/50
921/921 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0188 - val_loss: 0.0186
Epoch 7/50
921/921 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0184 - val_loss: 0.0184
Epoch 8/50
921/921 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0181 - val_loss: 0.0182
Epoch 9/50
921/921 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0179 - val_loss: 0.0181
Epoch 10/50
921/921 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0178 - val_loss: 0.0180
Epoch 11/50
921/921 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0177 - val_loss: 0.0179
Epoch 12/50
921/921 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step

In [27]:
# Save optimal model weights
ec_model.save_weights(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/ec_optimal_model.weights.h5'
)